# QTrack 2026 — Level 3: Quantum noise & error mitigation

**Tri-Co Hackathon · Bi-Co Quantum Computing Club**

**Mission:** Design a reproducible simulation experiment, quantify how noise degrades GHZ-state correlations, and mitigate readout errors.

**Prerequisites:** Levels 0–2 concepts, Python loops, NumPy arrays.  
**Deliverable:** A completed `.ipynb` notebook with your code, results, and explanations.

Follow the numbered tasks, implement the `TODO` functions, and use the self-checks to test your work.


## Getting started

Run the next cell. In Google Colab, upload this notebook and choose **Runtime → Run all** after completing your tasks. The first run may install Python packages using the internet. After installation, all circuit simulations run locally on the CPU.

**Bit order convention:** Qiskit writes the most significant bit on the **left** of a bitstring. For example `"10"` means qubit 1 is `1` and qubit 0 is `0`. Our checks follow this convention.

**If you get `ModuleNotFoundError`:** rerun the installation cell or restart the notebook kernel and rerun.

In [ ]:
# First run: installs dependencies only if missing (requires internet).
import sys, subprocess, importlib.util
if importlib.util.find_spec("qiskit") is None or importlib.util.find_spec("qiskit_aer") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "qiskit>=2.5,<3", "qiskit-aer>=0.17,<0.19", "numpy>=1.26,<3", "matplotlib>=3.8"])
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
print("Qiskit ready. Free local simulation only; no IBM account or paid hardware required.")

### How to work
1. Read the task and its expected result.
2. Replace `return None` with your implementation.
3. Run the **self-check** underneath. A passing check verifies an observable property but is not a complete grading decision.
4. Answer the **Explain** questions in Markdown, using your own words.
5. Run the notebook top to bottom before exporting. Do not delete your code, plots or outputs.

## Research question

**How do two-qubit gate errors and measurement errors affect GHZ-state measurements, and how much can a simple classical readout-mitigation procedure recover?**

We use a synthetic noise model, **not a physical-device calibration**. The baseline success metric is the observed probability of either all zeros or all ones. Ideal GHZ measurements have success 1 on this metric.

Readout mitigation uses a known symmetric bit-flip probability `p_readout` and matrix inversion. This corrects a particular simulated measurement error; it **cannot undo two-qubit gate errors** or guarantee a physical speedup.

**Worked example — a symmetric 1-bit readout error matrix:**

In [ ]:
p = 0.05
A1 = np.array([[1-p,p],[p,1-p]])
print(A1)
print('Columns describe the observed-bit probabilities conditional on the true bit.')

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

### Task 3A — Prepare GHZ

Build n=3 and n=4 GHZ circuits. **Expected:** ideally only `000...` and `111...` appear.

In [ ]:
def build_ghz(n: int = 3) -> QuantumCircuit:
    """Build an n-qubit GHZ state for n=3 or 4, without measurements."""
    # TODO: Implement this function. You may use Qiskit and NumPy.
    return None

In [ ]:
# Visible self-check (not a hidden grader)
if build_ghz(3) is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    p = Statevector.from_instruction(build_ghz(3)).probabilities_dict()
    assert p.get('000',0)>.49 and p.get('111',0)>.49
    print("Self-check passed.")

### Task 3B — Simulate noise

Run a local GHZ experiment with a two-qubit depolarizing error and a symmetric readout error.

In [ ]:
def noisy_counts(n: int = 3, p_cx: float = 0.04, p_readout: float = 0.05, shots: int = 4096, seed: int = 21) -> dict:
    """Simulate GHZ with depolarizing error on CX and independent symmetric measurement-bit errors."""
    # TODO: Implement this function. You may use Qiskit and NumPy.
    return None

In [ ]:
# Visible self-check (not a hidden grader)
if noisy_counts() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    t = noisy_counts(n=3,p_cx=.04,p_readout=.05,shots=512,seed=21)
    assert sum(t.values())==512 and all(len(k)==3 for k in t)
    print("Self-check passed.")

### Task 3C — Define the outcome metric

Compute `P(000…0)+P(111…1)` from counts. **Expected:** clean GHZ success approaches 1.

In [ ]:
def ghz_success(counts: dict, n: int) -> float:
    """Fraction of shots that return either n zeros or n ones."""
    total = sum(counts.values())
    return (counts.get("0" * n, 0) + counts.get("1" * n, 0)) / total

In [ ]:
# Visible self-check (not a hidden grader)
if ghz_success({"000":4,"111":4,"001":2},3) is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    assert abs(ghz_success({'000':4,'111':4,'001':2},3)-.8)<1e-12
    for counts, n, expected in (
        ({"0000": 3, "1111": 4, "0001": 3}, 4, 0.7),
        ({"000": 3, "001": 2}, 3, 0.6),
        ({"1111": 3, "0001": 2}, 4, 0.6),
        ({"001": 10}, 3, 0.0),
        ({"000": 0.4, "111": 0.4, "001": 0.2}, 3, 0.8),
    ):
        assert abs(ghz_success(counts, n) - expected) < 1e-12
    for counts in ({}, {"000": 0, "111": 0}):
        try:
            ghz_success(counts, 3)
        except ZeroDivisionError:
            pass
        else:
            raise AssertionError("zero total must not produce a probability")
    print("Self-check passed.")

### Task 3D — Mitigate readout error

Build the assignment matrix for independent bit-flip errors, invert it, clip small negative estimates and renormalize.

In [ ]:
def readout_mitigate(counts: dict, n: int, p_readout: float) -> dict:
    """Return estimated true bitstring probabilities after symmetric readout matrix inversion."""
    if not 0 <= p_readout <= 1 or p_readout == 0.5:
        raise ValueError("p_readout must be in [0, 1] and not 0.5")
    total = sum(counts.values())
    if total <= 0:
        raise ValueError("counts must have positive total")
    A1 = np.array([[1 - p_readout, p_readout], [p_readout, 1 - p_readout]])
    A = np.eye(1)
    for _ in range(n):
        A = np.kron(A, A1)
    states = [format(i, f"0{n}b") for i in range(2**n)]
    observed = np.array([counts.get(state, 0) / total for state in states])
    corrected = np.linalg.solve(A, observed)
    corrected = np.clip(corrected, 0, None)
    corrected /= corrected.sum()
    return {state: float(value) for state, value in zip(states, corrected)}

In [ ]:
# Visible self-check (not a hidden grader)
if readout_mitigate({"000":100,"111":100},3,0.0) is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    t=readout_mitigate({'000':100,'111':100},3,0.0)
    assert abs(sum(t.values())-1)<1e-10 and abs(t['000']-.5)<1e-10
    for n in (3, 4):
        size = 2**n
        states = [format(i, f"0{n}b") for i in range(size)]
        weights = np.arange(1, size + 1, dtype=float)
        counts = {
            states[i]: sum(
                (j + 1) * 19**(n - (i ^ j).bit_count())
                for j in range(size)
            )
            for i in range(size)
        }
        recovered = readout_mitigate(counts, n, 0.05)
        np.testing.assert_allclose(
            [recovered[s] for s in states], weights / weights.sum(),
            rtol=0, atol=1e-12,
        )
        assert set(recovered) == set(states)
        assert all(v >= 0 for v in recovered.values())
        assert abs(sum(recovered.values()) - 1) < 1e-12

        clipped = readout_mitigate({"0" * n: 100}, n, 0.05)
        expected = np.array([
            19**(n - s.count("1")) if s.count("1") % 2 == 0 else 0
            for s in states
        ], dtype=float)
        expected /= expected.sum()
        np.testing.assert_allclose(
            [clipped[s] for s in states], expected, rtol=0, atol=1e-12,
        )
        flipped = readout_mitigate({"1" * n: 100}, n, 1.0)
        assert abs(flipped["0" * n] - 1) < 1e-12

    for counts, p in (
        ({}, 0.05),
        ({"000": 0, "111": 0}, 0.05),
        ({"000": 100}, 0.5),
        ({"000": 100}, -0.1),
        ({"000": 100}, 1.1),
        ({"000": 100}, float("nan")),
    ):
        try:
            readout_mitigate(counts, 3, p)
        except ValueError:
            pass
        else:
            raise AssertionError("expected invalid-input rejection")
    print("Self-check passed.")

### Task 3E — Small reproducible study

Collect eight runs (n=3,4; CX error 0, .02, .05, .10), compare the metric before and after readout mitigation, and chart the result.

In [ ]:
def experiment_table() -> list:
    """Return rows of results for n=3,4 and p_cx=0,0.02,0.05,0.1 (readout error fixed to .05)."""
    # TODO: Implement this function. You may use Qiskit and NumPy.
    return None

In [ ]:
# Visible self-check (not a hidden grader)
if experiment_table() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    t=experiment_table()
    assert len(t)==8 and all(0<=r['raw_success']<=1.0 and 0<=r['mitigated_success']<=1.0 for r in t)
    print("Self-check passed.")

## Figure and discussion (required)

Create a chart of **raw** versus **mitigated** GHZ success against `p_cx`, one line per system size (n=3,4) if possible. Include axis labels, units and a legend. Then answer, in 180–300 words:

1. What happens as the CX error probability grows? How does it compare for n=3 and n=4?
2. Does correcting readout error also remove two-qubit gate errors? Explain.
3. How does the finite number of shots affect certainty? Give at least one limitation of your simulation.
4. Provide a testable next-step research hypothesis.

**Write your report below.**

In [ ]:
import matplotlib.pyplot as plt
if experiment_table() is not None:
    rows = experiment_table()
    for n in (3,4):
        these = [r for r in rows if r['n']==n]
        xs = [r['p_cx'] for r in these]
        plt.plot(xs,[r['raw_success'] for r in these],marker='o',label=f'n={n} raw')
        plt.plot(xs,[r['mitigated_success'] for r in these],marker='s',linestyle='--',label=f'n={n} mitigated')
    plt.xlabel('CX depolarizing error parameter')
    plt.ylabel('P(all 0s or all 1s)')
    plt.ylim(0,1.05)
    plt.title('GHZ success under synthetic gate and readout noise')
    plt.legend()
    plt.grid(alpha=.3)
    plt.show()
    for row in rows: print(row)
else:
    print('Finish 3A–3E to generate the experiment plot.')

## Completion checklist
- [ ] GHZ circuit supports 3 and 4 qubits
- [ ] Synthetic noise model implemented
- [x] Success fraction and matrix-inversion mitigation implemented
- [ ] Eight experiment rows reported, with plot
- [ ] Explanation, limitations and next-step hypothesis included
- [ ] Notebook runs top to bottom

**Optional research extension:** Explore shot counts 512, 2048, 8192 for one setting. Compare uncertainty across seeds. State clearly whether your mitigation improves *accuracy* or merely changes an estimator.

## AI Use Disclosure
Task 3C:
- OpenAI Codex (gpt-6.1-sol), with parallel AI analysis, implemented the GHZ success fraction and verified count and probability-weight cases.

Task 3D:
- OpenAI Codex (gpt-6.1-sol), with parallel AI analysis, implemented assignment-matrix readout mitigation and verified nonzero-noise recovery, clipping, normalization, and invalid-input rejection.